# Session 7 — From testing records to an SIR model

**ABM course · synthetic testing data · an unidentified pathogen**

We will use one simple epidemic model to connect individual test records with population-level dynamics. The notebook moves in three steps: people move between SIR states; equations describe those movements; and weekly testing records help estimate the model’s parameters.

The aim is to understand how a model represents social dynamics. This is an introduction, not a full epidemiology course. In the next part of the lesson, we can use the fitted SIR behavior as a reference for an agent-based model.

## 1. A contagious setting as state transitions

Think of each person as being in one of three states at a given time:

| State | Meaning in this model | At time zero |
|---|---|---:|
| **Susceptible (S)** | Can become infected. | Generally almost all the population |
| **Infectious (I)** | Has the infection and can pass it to others. | Generally very few cases |
| **Recovered (R)** | Is no longer infectious and is treated as immune in this model. | Generally zero |

People can move from **S to I** through contact with someone infectious, then from **I to R** as they recover. The model does not include death, births, or reinfection. Its population is closed, so the three groups always add up to (N):

\[
S + I + R = N.
\]

Here, “time zero” means the beginning of the model. These descriptions show a general starting pattern.

## 2. The mathematical representation

The differential equations describe how quickly each group changes over time. Read \(dS/dt\), for example, as “how quickly the susceptible group is changing.”

| Differential equation | How to read it |
|---|---|
| \(\displaystyle \frac{dS}{dt}=-\beta\frac{SI}{N}\) | As time passes (\(dt\)), the susceptible group gets smaller as infections move people from **S** to **I**. |
| \(\displaystyle \frac{dI}{dt}=\beta\frac{SI}{N}-\gamma I\) | As time passes (\(dt\)), the infectious group grows through new infections and shrinks through recoveries. Its change is the balance of those two flows. |
| \(\displaystyle \frac{dR}{dt}=\gamma I\) | As time passes (\(dt\)), the recovered group grows as infectious people recover. |

Here, \(N\) is the total population. The expression \(SI/N\) represents opportunities for contact between susceptible and infectious people in a well-mixed population. The parameters \(\beta\) and \(\gamma\) determine the rates of the two flows. Time is measured in **weeks** to match the testing summaries.

### What the parameters tell us

- **\(\beta\), transmission rate:** how quickly contact with infectious people creates new infections when almost everyone is susceptible.
- **\(\gamma\), recovery rate:** how quickly infectious people move to the recovered state. In this model, \(1/\gamma\) is the average time spent infectious.
- **\(R_0=\beta/\gamma\), basic reproduction number:** the model’s average number of new infections caused by one infectious person at the start, when nearly everyone is susceptible.

Under these assumptions, \(R_0>1\) means infections initially grow; \(R_0<1\) means they initially shrink. It is a summary of the model’s starting conditions, not a count of exactly how many people a particular person infects and not, by itself, an official epidemic classification.

Next we will estimate \(\beta\) and \(\gamma\) from the synthetic records. The estimates will describe this dataset under the SIR assumptions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import binom
from IPython.display import clear_output, display
import ipywidgets as widgets

plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

### Explore the infection term interactively

Choose susceptible and infectious counts. The widget calculates recovered people as the remainder, so \(S+I+R=N\) always holds. Notice that knowing \(S\) alone is not enough to calculate \(SI/N\).

Try \(S=9{,}000, I=1{,}000\), then \(S=1{,}000, I=9{,}000\). Both have \(R=0\). Then keep \(S=1{,}000\) and change \(I\) to \(1{,}000\): the remaining people are in \(R\), and the infection term changes.

In [ ]:
N = 10_000
susceptible_slider = widgets.IntSlider(
    value=9_000, min=0, max=N, step=500,
    description="Susceptible S", continuous_update=False,
)
infectious_slider = widgets.IntSlider(
    value=1_000, min=0, max=N, step=500,
    description="Infectious I", continuous_update=False,
)
infectious_slider.max = N - susceptible_slider.value

# Keep S + I within the population; R is the remainder.
def limit_infectious(change):
    infectious_slider.max = N - change["new"]
    if infectious_slider.value > infectious_slider.max:
        infectious_slider.value = infectious_slider.max

susceptible_slider.observe(limit_infectious, names="value")

def show_infection_term(S, I):
    R = N - S - I
    term = S * I / N
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(8, 2.8))
    values = [S, I, R]
    bars = ax.bar(["Susceptible S", "Infectious I", "Recovered R"], values,
                  color=["#087e8b", "#e07a32", "#9aa5ab"])
    ax.set_ylim(0, N)
    ax.set_ylabel("People")
    ax.set_title(f"Population split (N = {N:,})")
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, value + N*0.025,
                f"{value:,}", ha="center")
    ax.grid(axis="y", alpha=0.2)
    plt.show()
    print(f"Scaled infection term, S × I / N = {term:,.0f}")
    print(f"The infection flow in the equation is β × {term:,.0f} people per week.")

controls = widgets.VBox([susceptible_slider, infectious_slider])
result = widgets.interactive_output(show_infection_term,
                                   {"S": susceptible_slider, "I": infectious_slider})
display(controls, result)

## 3. From test records to SIR parameters

The CSV contains one row for each completed test. It follows this teaching scenario:

- The population is **10,000**; this denominator is set here, not repeated in the CSV.
- Week 1 is a random baseline sample of 100 tests, taken before the outbreak is seeded.
- From week 2 onward, there are 80 fresh random surveillance tests each week. Up to 20 additional kits may be used for follow-up tests of earlier positive cases, four weeks later. When more than 20 follow-ups are due, 20 are selected randomly. A person who does not return uses no kit, and the sample is not backfilled.

We use the random **surveillance** sample to estimate the share infectious. Follow-ups are targeted at earlier positive cases, so we use them separately to estimate recovery.

### Step 1: open the CSV

In [ ]:
POPULATION = N
TESTS_URL = "https://raw.githubusercontent.com/DACSS-Complexity/s7_epidemics/main/session7_synthetic_tests.csv"

records = pd.read_csv(TESTS_URL, parse_dates=["date"])
records.head()

### Step 2: keep the surveillance sample

A surveillance sample is meant to represent the population. A follow-up sample is different: it deliberately retests people who tested positive earlier. Combining the two would make the positivity rate look higher than the population sample suggests.

In [ ]:
surveillance = records.loc[records.test_type.eq("surveillance")].copy()
surveillance["positive"] = surveillance.result.eq("positive").astype(int)

print(f"Surveillance tests: {len(surveillance):,}")
print(f"Follow-up tests: {(records.test_type == 'followup').sum():,}")

### Step 3: summarize tests by week

For each week, count how many surveillance tests were completed and how many were positive. Keeping both counts matters: 10 positives out of 20 tests tells a different sampling story from 10 out of 80.

In [ ]:
weekly = surveillance.groupby("week").agg(
    date=("date", "min"),
    tests=("test_id", "size"),
    positives=("positive", "sum"),
)

weekly.head(8)

### Step 4: turn positivity into an SIR observation

The **positivity rate** is the share of surveillance tests that are positive. If the sample is random and the tests are accurate, it gives a sample-based estimate of the infectious share, \(I/N\). Multiplying by the population gives an approximate infectious count; it is not a census of all 10,000 people.

In [ ]:
weekly["positivity"] = weekly.positives / weekly.tests
weekly["infectious_share_estimate"] = weekly.positivity
weekly["infectious_count_estimate"] = weekly.positivity * POPULATION

weekly[["tests", "positives", "positivity", "infectious_count_estimate"]].round(3)

The rising positivity in this synthetic dataset gives us a pattern to compare with the SIR model. Week 1 is the pre-outbreak baseline, so we use weeks 2 onward when fitting the outbreak trajectory. A zero in a small weekly sample would mean “no positives observed in this sample,” not “no infectious people exist.”

In [ ]:
ax = weekly.positivity.mul(100).plot(marker="o", color="#087e8b")
ax.set(title="Weekly surveillance positivity", xlabel="Week", ylabel="Positive tests (%)")
ax.grid(alpha=0.2)
plt.show()

### Step 5: use follow-up tests to estimate \(\gamma\)

In this **synthetic exercise**, a negative follow-up four weeks after a positive surveillance test marks that person as recovered by the retest; a positive follow-up means the infection is still detected. We compare each person’s first positive surveillance test with a follow-up exactly four weeks later. Missing follow-ups are not counted as recoveries.

The model assumes a constant recovery rate. Under that assumption, the chance of still being infectious after \(d\) weeks is \(e^{-\gamma d}\). Since these follow-ups are four weeks apart, the observed share still positive lets us solve for \(\gamma\).

In [ ]:
# Find each person's first positive surveillance test.
first_positive = records.loc[
    records.test_type.eq("surveillance") & records.result.eq("positive"),
    ["person_id", "week"],
].sort_values("week").drop_duplicates("person_id")
first_positive = first_positive.rename(columns={"week": "positive_week"})

# Link follow-ups to that first positive test.
followups = records.loc[records.test_type.eq("followup")].rename(
    columns={"week": "followup_week"}
)
returned = followups.merge(first_positive, on="person_id", how="inner")
returned = returned.loc[returned.followup_week - returned.positive_week == 4].copy()

returned.result.value_counts()

In [ ]:
still_positive_share = returned.result.eq("positive").mean()
FOLLOWUP_WEEKS = 4
GAMMA_ESTIMATE = -np.log(still_positive_share) / FOLLOWUP_WEEKS
AVERAGE_INFECTIOUS_WEEKS = 1 / GAMMA_ESTIMATE

print(f"Four-week follow-ups used: {len(returned)}")
print(f"Estimated gamma: {GAMMA_ESTIMATE:.3f} per week")
print(f"Model-implied average infectious period: {AVERAGE_INFECTIOUS_WEEKS:.1f} weeks")

This estimate relies on the model’s constant-recovery assumption and on the synthetic follow-up rule. In real data, a negative test is evidence from a test, not automatic proof of when someone stopped being infectious.

### Step 6: prepare the surveillance counts for fitting \(\beta\)

We will compare the SIR model’s expected infectious share with the observed number of positives out of the tested sample each week. The fit starts in week 2. In the data fit, week 1 is a pre-outbreak baseline and the fit starts at week 2. The fitting procedure estimates the infectious count at that point from the surveillance data. For simplicity, it assumes \(R=0\) at the start of the fitted period.

In [ ]:
observed = weekly.loc[weekly.index >= 2]
observed_weeks = observed.index.to_numpy(dtype=int)
observed_n = observed.tests.to_numpy(dtype=int)
observed_positive = observed.positives.to_numpy(dtype=int)

print(f"Weeks used in fit: {len(observed)} (weeks {observed_weeks[0]}–{observed_weeks[-1]})")

### Step 7: describe one week of SIR movement

The differential equations describe continuous change. To connect them to weekly observations, the function below advances the compartments one week at a time. It uses the probability of infection or recovery over a week, based on \(\beta\) and \(\gamma\).

In [ ]:
def sir_one_week(susceptible, infectious, recovered, beta):
    new_infections = susceptible * (-np.expm1(-beta * infectious / POPULATION))
    new_recoveries = infectious * (-np.expm1(-GAMMA_ESTIMATE))

    susceptible -= new_infections
    infectious += new_infections - new_recoveries
    recovered += new_recoveries
    return susceptible, infectious, recovered

Now apply that one-week movement repeatedly to get the model’s expected S, I, and R for the observed weeks.

In [ ]:
def sir_states_by_week(beta, initial_infectious):
    susceptible = POPULATION - initial_infectious
    infectious = initial_infectious
    recovered = 0.0
    states = []
    previous_week = int(observed_weeks[0])

    for week in observed_weeks:
        for _ in range(int(week - previous_week)):
            susceptible, infectious, recovered = sir_one_week(
                susceptible, infectious, recovered, beta
            )
        states.append((susceptible, infectious, recovered))
        previous_week = int(week)

    return np.asarray(states)

### Step 8: let the data estimate \(\beta\)

The model predicts a probability of testing positive in each weekly surveillance sample: its infectious share, \(I/N\). We score how well those probabilities match the observed positive and negative counts, then let an optimizer search for the \(\beta\) and starting infectious count that best fit the data.

This is a statistical fit, not a direct reading of \(\beta\) from any single CSV column.

In [ ]:
def negative_log_likelihood(log_parameters):
    beta, initial_infectious = np.exp(log_parameters)
    states = sir_states_by_week(beta, initial_infectious)
    model_positive_share = np.clip(states[:, 1] / POPULATION, 1e-12, 1 - 1e-12)

    # Each week's positives are counted out of that week's surveillance tests.
    return -binom.logpmf(observed_positive, observed_n, model_positive_share).sum()

In [ ]:
fit = minimize(
    negative_log_likelihood,
    x0=np.log([0.5, 20]),
    method="L-BFGS-B",
    bounds=[(np.log(0.001), np.log(2.0)),
            (np.log(0.1), np.log(POPULATION - 1))],
)

BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE = np.exp(fit.x)
R0_ESTIMATE = BETA_ESTIMATE / GAMMA_ESTIMATE

print(f"Estimated beta: {BETA_ESTIMATE:.3f} per week")
print(f"Estimated gamma: {GAMMA_ESTIMATE:.3f} per week")
print(f"Estimated R0 = beta / gamma: {R0_ESTIMATE:.2f}")
print(f"Estimated infectious people at start of week 2: {INITIAL_INFECTIOUS_ESTIMATE:.0f}")

### Step 9: compare the fitted SIR curve with the sample

The dots show the weekly share of surveillance tests that were positive. The line shows the infectious share produced by the fitted SIR model. The model is trying to follow the overall pattern; it does not need to pass through every sample point.

In [ ]:
fitted_states = sir_states_by_week(BETA_ESTIMATE, INITIAL_INFECTIOUS_ESTIMATE)

plt.scatter(observed_weeks, 100 * observed_positive / observed_n,
            color="#a7bdc5", label="Observed surveillance positivity")
plt.plot(observed_weeks, 100 * fitted_states[:, 1] / POPULATION,
         color="#087e8b", linewidth=2, label="Fitted SIR infectious share")
plt.xlabel("Week")
plt.ylabel("Share (%)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### What we learned from the fit

The testing sample gives evidence about how the infectious share changes over time. The follow-up sample gives evidence about how quickly detected infections move toward recovery under the model’s assumptions. Together, they let us estimate \(\beta\), \(\gamma\), and the implied \(R_0\).

These are **population-level, model-based estimates**. The SIR model has no contact tracing information, so it cannot tell us who infected whom or identify high-contact individuals. The estimates give us a data-informed reference for the next step: parameterizing a simple ABM and then exploring clearly stated social scenarios.

### Assumptions to keep in mind

This teaching fit assumes random surveillance sampling, accurate tests, a closed population, homogeneous mixing, lasting immunity, and a constant recovery rate. A negative four-week follow-up marks recovery in this synthetic exercise. The SIR model summarizes population behavior; it does not identify individual contact patterns. The estimates are conditional on these assumptions and on this small synthetic sample.

## Student exercises

1. In your own words, describe the three SIR states and the two movements between them.
2. Why do we use surveillance tests, rather than combine surveillance and follow-up tests, to estimate weekly positivity?
3. Explain how a four-week follow-up result informs \(\gamma\) in this synthetic exercise.
4. What does the fitted \(R_0\) summarize? What information does it not give us about particular people?
5. Name one assumption in this notebook that could be changed or supported with additional data before carrying the estimates into an ABM.